# DPOによる強化学習をするサンプル

Qwen-3 0.6Bモデルを使用して、DPO(Direct Preference Optimization) を用いた強化学習を行うサンプルコードです。

DPOでは報酬モデルを利用せず、ユーザが定義した好ましい応答と好ましくない応答の生成確率そのものを比較して最適化します。RLHFと比較して、報酬モデルの学習が不要なため、計算コストが削減され、学習が安定するという利点があります。

- RLHF: 「SFT -> 報酬モデル学習 -> PPOによる方針更新」の事前学習と微調整の3段階のプロセスが必要
- DPO: 「SFT -> DPOによる方針更新」の2段階のプロセスで済む

## データセットの形式

DPOで用いられるデータセットの形式が以下です。`chosen`が好ましい応答、`rejected`が好ましくない応答です。

```json
{
	"conversations": [
		{
			"from": "human",
			"value": "最近、私は"
		},
		{
			"from": "gpt",
			"value": "すみません。よく聞き取れませんでした。"
		},
		{
			"from": "human",
			"value": "すみません、偶然です。最近数学をよくやっていると言っていたんです。比の概念を数学的な文脈で説明できますか。"
		}
	],
	"chosen": "比は数の研究です。それは互いに比較することができる2つの数の間の関係であり、比は実際には関係そのものです。",
	"rejected": "はい、もちろんです。  まず、あなたが比率について学ぶ理由は何ですか？",
}
```

In [1]:
%pip install trl -q

In [2]:
import torch
from tqdm.auto import tqdm
from datetime import datetime
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig

In [3]:
cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
use_bf16 = cap[0] >= 8
dtype = torch.bfloat16 if use_bf16 else torch.float16
bf16 = use_bf16
fp16 = not use_bf16

print(f"CUDA Device Capability: {cap}, Using bf16: {bf16}, fp16: {fp16}")

CUDA Device Capability: (7, 5), Using bf16: False, fp16: True


In [4]:
model_id = "Qwen/Qwen3-0.6B"

print(f"Using model: {model_id}")

Using model: Qwen/Qwen3-0.6B


## データセットの準備

In [ ]:
# サンプルデータセットを読み込み
dataset = load_dataset("llm-jp/hh-rlhf-12k-ja", split="train")

In [ ]:
# データセットの件数と学習データの割合を設定。サンプルでは10,000件だが時間がかかりすぎるので1,000件に制限
dataset_count = 1000

# helpful-baseデータセットの最初のdataset_count件を使用
dataset = dataset.filter(lambda x: x["source"] == "helpful-base")

dataset = dataset.select(range(dataset_count))

In [ ]:
# データの内容を確認
dataset[0]

{'conversations': [{'from': 'human', 'value': '最近、私は'},
  {'from': 'gpt', 'value': 'すみません。よく聞き取れませんでした。'},
  {'from': 'human',
   'value': 'すみません、偶然です。最近数学をよくやっていると言っていたんです。比の概念を数学的な文脈で説明できますか。'}],
 'chosen': '比は数の研究です。それは互いに比較することができる2つの数の間の関係であり、比は実際には関係そのものです。',
 'rejected': 'はい、もちろんです。  まず、あなたが比率について学ぶ理由は何ですか？',
 'source': 'helpful-base'}

## データセットの形式

In [ ]:
# DPO用にデータセットの形式を変換する関数を定義
def convert_for_dpo(example: dict[str, any]) -> dict[str, any]:
    def convert_role(from_str: str) -> str:
        return "user" if from_str == "human" else "assistant"
    # 1) 会話履歴を role/content 形式に変換
    history = [
        {
            "role": convert_role(turn["from"]),
            "content": turn["value"],
        }
        for turn in example["conversations"]
    ]

    # 2) chosen / rejected をそれぞれ履歴 + 最後の assistant 応答にする
    chosen_msgs = history + [
        {
            "role": "assistant",
            "content": example["chosen"],
        }
    ]
    rejected_msgs = history + [
        {
            "role": "assistant",
            "content": example["rejected"],
        }
    ]
    return {
        "chosen": chosen_msgs,
        "rejected": rejected_msgs
    }

In [ ]:
# データセットをDPO用に変換
formatted_dataset = dataset.map(
  convert_for_dpo, 
  remove_columns=dataset.column_names
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

### 無料のColab環境で実行する場合の設定

In [ ]:
max_chars = 20

def calc_total_chars(messages):
    # [{'role':..., 'content':...}, ...] の content を全部連結
    return sum(len(m["content"]) for m in messages)

def filter_dpo(example):
    chosen_len = calc_total_chars(example["chosen"])
    rejected_len = calc_total_chars(example["rejected"])

    # どちらかが長すぎる場合は除外
    return (chosen_len <= max_chars) and (rejected_len <= max_chars)

if fp16:
    print("FP16 mode: Filtering dataset for max chars =", max_chars)
    formatted_dataset = formatted_dataset.filter(filter_dpo)
    print(len(formatted_dataset))

FP16 mode: Filtering dataset for max chars = 20


Filter:   0%|          | 0/375 [00:00<?, ? examples/s]

0


## 学習器の設定

In [ ]:
per_device_train_batch_size = 1 if fp16 else 2
gradient_accumulation_steps = 8  # 実効バッチを稼ぐ
warmup_ratio = 0.03
num_train_epochs = 10 if bf16 else 1  # 無料版は時間がかかるので1epoch

current_date = datetime.now().strftime("%Y%m%d")
output_dir_name = f"../results/continued-pretrain_{current_date}"

# お試しのためほぼデフォルトの設定を使用。他の設定は以下参照
# https://huggingface.co/docs/trl/dpo_trainer#trl.DPOConfig
args = DPOConfig(
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    fp16=fp16,
    bf16=bf16,
    logging_steps=10,
    seed=42,
    output_dir=output_dir_name,
    report_to="tensorboard",
    save_total_limit=1,
    beta=0.1,
    warmup_ratio=warmup_ratio,
)

: 

In [ ]:
dpo_trainer = DPOTrainer(
    model=model_id,
    args=args,
    train_dataset=formatted_dataset
)

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1566: UserWarning: Current model requires 256 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


: 

: 

In [ ]:
dpo_trainer.train()

NameError: name 'dpo_trainer' is not defined